In [ ]:
from fastai.vision.all import *
from torch.nn import *
from re import compile


def Flat(a):
    return a.ravel().tolist()


def Unzip(a, n):
    return np.array(a, "O").reshape(-1, n)


def Rand(a, c=1):
    return (np.random.rand(len(a)) ** c * a).astype(int)


def Sort(a):
    a.sort()
    return a


class Dat:
    r = compile("[\u4e00-\u9fa5]")
    m = {}

    def __init__(s, a, b=4):
        a = [loads(i) for i in open(f"iflytek/{a}.json")]
        s.y = torch.tensor([int(a["label"]) for a in a], device="cuda")
        if b:
            a = [[s.m.setdefault(a, len(s.m)) for a in s.r.findall(a["sentence"])] for a in a]
        else:
            a = [[s.m[a] for a in a["sentence"] if a in s.m] for a in a]
        s.x = torch.tensor([a for a in a for a in a], device="cuda")
        s.l = np.array([len(a) for a in a])
        s.L = np.r_[0, s.l].cumsum()
        s.b = b

    def __len__(s):
        return len(s.y) // s.b if s.b else 1

    def __iter__(s):
        if s.b:
            for t in range(len(s)):
                i = np.random.choice(len(s.y), s.b, 0)
                l = s.l[i]
                t = Rand(l, 0.5) + 1
                yield [s.x[torch.tensor(np.concatenate([L + Sort(np.random.choice(l, t, 0)) for L, l, t in zip(s.L[i], l, t)]), device="cuda")], t], s.y[
                    torch.tensor(i, device="cuda")
                ]
        else:
            yield [s.x, s.l], s.y


class F(Sequential):
    def __init__(s, f, *a):
        super().__init__(*a)
        s.f = f

    def forward(s, *x):
        return s.f(super().forward, *x)


class Red(Sequential):
    def forward(s, x):
        x, l = x
        while len(x) > len(l):
            t = Rand(l + 1) & 0xFFFFFFFE
            l = np.stack([t, l & 1, (l & 0xFFFFFFFE) - t], 1)
            x = Unzip(x.split(Flat(l)), 3)
            l[:, ::2] >>= 1
            x[:, ::2] = Unzip(super().forward(torch.cat(Flat(x[:, ::2])).view(-1, x[0, 0].size(1) * 2)).split(Flat(l[:, ::2])), 2)
            x = torch.cat(Flat(x))
            l = l.sum(1)
        return x

In [ ]:
m = Learner(
    DataLoaders(Dat("train"), Dat("dev", 0)),
    Sequential(
        F(lambda f, x: [f(x[0]), x[1]], Embedding(len(Dat.m), 8)),
        Red(Linear(16, 8), F(lambda f, x: f(x) if len(x) > 1 else x, BatchNorm1d(8, track_running_stats=0)), Swish()),
        Linear(8, 8, 0),
        BatchNorm1d(8, track_running_stats=0),
        Swish(),
        Linear(8, 119),
    ).cuda(),
    CrossEntropyLoss(),
    metrics=accuracy,
    cbs=ShowGraphCallback,
)
m.lr_find()

In [ ]:
m.fit_one_cycle(999, _[0], 9, 99, .01)